In [ ]:
# Import librerie necessarie
import sys
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors
import folium
from sklearn.cluster import KMeans
from sklearn.neighbors import BallTree
from sklearn.metrics.pairwise import haversine_distances
from scipy.spatial import ConvexHull
from shapely.geometry import Polygon

# Aggiungi la root del progetto al path per gli import
sys.path.insert(0, os.path.abspath('../..'))

# Import configurazioni
from app.core.config import settings, IMMOBILI_QUOTAZIONE_PATH
from app.data.loaders import load_and_merge_data
from amenities_config import CATEGORY_AMENITIES

# Costanti
EARTH_RADIUS_KM = 6371.0
seed = 42

# Mappatura categoria -> amenities per l'identificazione dei tag
category_amenities = {}
for category, amenities in CATEGORY_AMENITIES.items():
    category_amenities[category] = [(amenity, amenity) for amenity in amenities]

print("✓ Import completati")
print(f"Categorie disponibili: {list(CATEGORY_AMENITIES.keys())}")

# Clustering Analysis for Real Estate Data

This notebook performs clustering analysis on real estate data, generating datasets for cluster metadata, quotazioni, and amenity counts at various distances.

It creates three datasets:
1. Cluster metadata (id, centroid, OMI zone, area, density)
2. Quotazioni per gruppo catastale per cluster
3. Amenity counts at different distances per cluster

In [ ]:
def build_spatial_index(pois_by_category: dict) -> tuple:
    """
    Costruisce un BallTree per ricerca spaziale efficiente dei POI.
    Ritorna l'indice e la lista di tutti i POI con le loro coordinate.
    """
    all_pois = []
    for category, subcategories in pois_by_category.items():
        if isinstance(subcategories, dict):
            # Struttura annidata: categoria -> sottocategoria -> lista POI
            for subcategory, pois_list in subcategories.items():
                if isinstance(pois_list, list):
                    all_pois.extend(pois_list)
        elif isinstance(subcategories, list):
            # Struttura semplice: categoria -> lista POI
            all_pois.extend(subcategories)
    
    if not all_pois:
        return None, []
    
    # Converti coordinate in radianti per BallTree (usa distanza haversine)
    coords_rad = np.radians([[p['lat'], p['lon']] for p in all_pois])
    tree = BallTree(coords_rad, metric='haversine')
    
    return tree, all_pois

In [ ]:
def count_amenities_at_distances(centroids: np.ndarray, pois_by_category: dict, 
                                 distances_m: list = [250, 500, 1000, 2000]) -> pd.DataFrame:
    """
    Per ogni cluster e categoria, conta il numero di amenity a diverse distanze dal centroide.
    
    Args:
        centroids: Array (n_clusters, 2) con lat/lon dei centroidi
        pois_by_category: Dict con categoria -> sottocategoria -> lista POI
        distances_m: Lista di distanze in metri per i conteggi
    
    Returns:
        DataFrame con colonne: cluster_id, category, amenity, count_250, count_500, count_1000, count_2000
    """
    results = []
    n_clusters = len(centroids)
    
    # Converti distanze da metri a radianti
    distances_rad = [d / 1000.0 / EARTH_RADIUS_KM for d in distances_m]
    
    for category, subcategories in pois_by_category.items():
        if isinstance(subcategories, dict):
            # Struttura annidata: categoria -> sottocategoria -> lista POI
            for subcategory, pois_list in subcategories.items():
                if not isinstance(pois_list, list) or not pois_list:
                    continue
                
                # Organizza POI per tipo di amenity
                pois_by_amenity = {}
                for poi in pois_list:
                    # Usa la sottocategoria come amenity type se non ci sono tag
                    if 'tags' not in poi:
                        amenity_found = subcategory
                    else:
                        tags = json.loads(poi['tags'])
                        
                        # Identifica l'amenity per questa categoria
                        amenity_found = None
                        for amenity_key, amenity_value in category_amenities.get(category, []):
                            if (amenity_value == '*' and amenity_key in tags) or tags.get(amenity_key) == amenity_value:
                                amenity_found = tags.get(amenity_key) if amenity_value == '*' else amenity_value
                                break
                    
                    if amenity_found:
                        if amenity_found not in pois_by_amenity:
                            pois_by_amenity[amenity_found] = []
                        pois_by_amenity[amenity_found].append(poi)
                
                # Per ogni tipo di amenity nella categoria
                for amenity, amenity_pois in pois_by_amenity.items():
                    # Costruisci coordinate POI
                    poi_coords_rad = np.radians([[p['lat'], p['lon']] for p in amenity_pois])
                    
                    # Per ogni cluster
                    for cluster_id in range(n_clusters):
                        centroid_rad = np.radians([[centroids[cluster_id][0], centroids[cluster_id][1]]])
                        
                        # Calcola distanze
                        distances = haversine_distances(centroid_rad, poi_coords_rad)[0]
                        
                        # Conta amenity a ogni distanza
                        counts = {}
                        for dist_m, dist_rad in zip(distances_m, distances_rad):
                            count = np.sum(distances <= dist_rad)
                            counts[f'count_{dist_m}'] = count
                        
                        results.append({
                            'cluster_id': cluster_id,
                            'category': category,
                            'amenity': amenity,
                            **counts
                        })
        elif isinstance(subcategories, list):
            # Struttura semplice: categoria -> lista POI
            if not subcategories:
                continue
            
            # Organizza POI per tipo di amenity
            pois_by_amenity = {}
            for poi in subcategories:
                # Usa la categoria come amenity type se non ci sono tag
                if 'tags' not in poi:
                    amenity_found = category
                else:
                    tags = json.loads(poi['tags'])
                    
                    # Identifica l'amenity per questa categoria
                    amenity_found = None
                    for amenity_key, amenity_value in category_amenities.get(category, []):
                        if (amenity_value == '*' and amenity_key in tags) or tags.get(amenity_key) == amenity_value:
                            amenity_found = tags.get(amenity_key) if amenity_value == '*' else amenity_value
                            break
                
                if amenity_found:
                    if amenity_found not in pois_by_amenity:
                        pois_by_amenity[amenity_found] = []
                    pois_by_amenity[amenity_found].append(poi)
            
            # Per ogni tipo di amenity nella categoria
            for amenity, amenity_pois in pois_by_amenity.items():
                # Costruisci coordinate POI
                poi_coords_rad = np.radians([[p['lat'], p['lon']] for p in amenity_pois])
                
                # Per ogni cluster
                for cluster_id in range(n_clusters):
                    centroid_rad = np.radians([[centroids[cluster_id][0], centroids[cluster_id][1]]])
                    
                    # Calcola distanze
                    distances = haversine_distances(centroid_rad, poi_coords_rad)[0]
                    
                    # Conta amenity a ogni distanza
                    counts = {}
                    for dist_m, dist_rad in zip(distances_m, distances_rad):
                        count = np.sum(distances <= dist_rad)
                        counts[f'count_{dist_m}'] = count
                    
                    results.append({
                        'cluster_id': cluster_id,
                        'category': category,
                        'amenity': amenity,
                        **counts
                    })
    
    return pd.DataFrame(results)

In [ ]:
def save_poi_distances_per_cluster(centroids: np.ndarray, pois_by_category: dict, 
                                   max_distance_m: int = 2000) -> dict:
    """
    Per ogni cluster, categoria e amenity, salva id e distanza di ogni POI rispetto al centroide,
    ordinati per distanza crescente, escludendo quelli oltre max_distance_m.
    
    Args:
        centroids: Array (n_clusters, 2) con lat/lon dei centroidi
        pois_by_category: Dict con categoria -> sottocategoria -> lista POI
        max_distance_m: Distanza massima in metri (default 2000)
    
    Returns:
        Dict strutturato come cluster_id -> category -> amenity -> list of (poi_id, distance_m)
    """
    results = {}
    n_clusters = len(centroids)
    
    # Converti distanza massima da metri a radianti
    max_distance_rad = max_distance_m / 1000.0 / EARTH_RADIUS_KM
    
    for category, subcategories in pois_by_category.items():
        if isinstance(subcategories, dict):
            # Struttura annidata: categoria -> sottocategoria -> lista POI
            for subcategory, pois_list in subcategories.items():
                if not isinstance(pois_list, list) or not pois_list:
                    continue
                
                # Organizza POI per tipo di amenity
                pois_by_amenity = {}
                for poi in pois_list:
                    if 'tags' not in poi:
                        amenity_found = subcategory
                    else:
                        tags = json.loads(poi['tags'])
                        
                        amenity_found = None
                        for amenity_key, amenity_value in category_amenities.get(category, []):
                            if (amenity_value == '*' and amenity_key in tags) or tags.get(amenity_key) == amenity_value:
                                amenity_found = tags.get(amenity_key) if amenity_value == '*' else amenity_value
                                break
                    
                    if amenity_found:
                        if amenity_found not in pois_by_amenity:
                            pois_by_amenity[amenity_found] = []
                        pois_by_amenity[amenity_found].append(poi)
                
                # Per ogni tipo di amenity nella categoria
                for amenity, amenity_pois in pois_by_amenity.items():
                    # Costruisci coordinate POI
                    poi_coords_rad = np.radians([[p['lat'], p['lon']] for p in amenity_pois])
                    
                    # Per ogni cluster
                    for cluster_id in range(n_clusters):
                        if cluster_id not in results:
                            results[cluster_id] = {}
                        if category not in results[cluster_id]:
                            results[cluster_id][category] = {}
                        
                        centroid_rad = np.radians([[centroids[cluster_id][0], centroids[cluster_id][1]]])
                        
                        # Calcola distanze
                        distances_rad = haversine_distances(centroid_rad, poi_coords_rad)[0]
                        distances_m = distances_rad * EARTH_RADIUS_KM * 1000  # Converti in metri
                        
                        # Filtra POI entro max_distance_m e ordina per distanza
                        poi_distances = []
                        for i, dist_m in enumerate(distances_m):
                            if dist_m <= max_distance_m:
                                poi_id = amenity_pois[i].get('id', f"poi_{i}")
                                poi_distances.append((poi_id, dist_m))
                        
                        # Ordina per distanza crescente
                        poi_distances.sort(key=lambda x: x[1])
                        
                        results[cluster_id][category][amenity] = poi_distances
    
    return results

In [ ]:
def load_immobili_and_prepare_coords() -> tuple:
    """
    Carica il dataset degli immobili e prepara le coordinate per il clustering.
    Ritorna (coords, immobili_with_omi, quotazioni_df).
    """
    # Carica il dataset degli immobili usando percorso assoluto
    base_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
    immobili_path = os.path.join(base_path, settings.DATASET_FULL)
    immobili_df = load_and_merge_data(immobili_path)
    
    if immobili_df is None:
        print("Errore nel caricamento del dataset degli immobili.")
        return None, None, None
    
    print(f"Dataset caricato con successo: {len(immobili_df)} righe.")
    
    # Carica il dataset delle quotazioni
    quotazioni_path = os.path.join(base_path, IMMOBILI_QUOTAZIONE_PATH)
    quotazioni_df = pd.read_csv(quotazioni_path, sep=';', dtype={'id': str})
    print(f"Dataset quotazioni caricato: {len(quotazioni_df)} righe.")
    
    # Merge con le quotazioni (join su id)
    immobili_df = immobili_df.merge(
        quotazioni_df[['id', 'gruppo_catastale', 'quotazione_immobiliare_mq']], 
        on='id', 
        how='left'
    )
    
    # Assicurati che latitudine e longitudine siano numeriche
    immobili_df['latitudine'] = pd.to_numeric(immobili_df['latitudine'], errors='coerce')
    immobili_df['longitudine'] = pd.to_numeric(immobili_df['longitudine'], errors='coerce')
    immobili_df = immobili_df.dropna(subset=['latitudine', 'longitudine'])
    
    # Shuffling degli immobili prima del partizionamento
    immobili_df = immobili_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    # Carica il GeoJSON delle zone OMI
    omi_geojson_path = os.path.join(base_path, settings.ZONE_OMI_GEOJSON)
    omi_gdf = gpd.read_file(omi_geojson_path)
    
    # Crea GeoDataFrame per gli immobili
    immobili_gdf = gpd.GeoDataFrame(immobili_df, geometry=gpd.points_from_xy(immobili_df.longitudine, immobili_df.latitudine))
    immobili_gdf.set_crs('EPSG:4326', inplace=True)
    
    # Join spaziale per assegnare CODZONA agli immobili
    immobili_with_omi = gpd.sjoin(immobili_gdf, omi_gdf[['CODZONA', 'geometry']], how='left', predicate='within')
    
    # Se un immobile cade in più zone, prendi la prima
    immobili_with_omi = immobili_with_omi[~immobili_with_omi.index.duplicated(keep='first')]
        
    # Rimuovi immobili senza zona OMI
    immobili_with_omi = immobili_with_omi.dropna(subset=['CODZONA'])
    
    print(f"Immobili con zona OMI: {len(immobili_with_omi)}")
    
    # Coordinate per clustering (solo immobili con zona OMI)
    coords = immobili_with_omi[['latitudine', 'longitudine']].values
    
    return coords, immobili_with_omi, quotazioni_df

In [ ]:
def load_pois_data(pois_file: str = '../01_pois/pois_by_category.json') -> tuple:
    """
    Carica il file pois_by_category.json e costruisce l'indice spaziale.
    Ritorna (tree, all_pois, pois_by_category).
    """
    with open(pois_file, 'r') as f:
        pois_by_category = json.load(f)
    
    tree, all_pois = build_spatial_index(pois_by_category)
    return tree, all_pois, pois_by_category

In [ ]:
def create_cluster_metadata(clusters_df: pd.DataFrame, immobili_with_omi: gpd.GeoDataFrame, 
                           labels: np.ndarray, coords: np.ndarray, n_clusters: int) -> pd.DataFrame:
    """
    Crea il dataset con metadata dei cluster includendo:
    - cluster_id, lat, lon del centroide
    - fascia_omi (maggioritaria, solo lettera)
    - num_immobili
    - superficie_km2
    - densita_immobili_per_km2
    """
    metadata_list = []
    
    for i in range(n_clusters):
        cluster_mask = labels == i
        cluster_immobili = immobili_with_omi[cluster_mask]
        cluster_points = coords[cluster_mask]
        
        # Fascia OMI maggioritaria (solo lettera)
        fascia_counts = cluster_immobili['CODZONA'].str[0].value_counts()
        majority_fascia = fascia_counts.idxmax() if not fascia_counts.empty else None
        
        # Numero immobili
        num_immobili = len(cluster_immobili)
        
        # Calcola superficie usando ConvexHull
        superficie_km2 = 0.0
        densita = 0.0
        
        if len(cluster_points) >= 3:
            try:
                hull = ConvexHull(cluster_points)
                hull_points = cluster_points[hull.vertices]
                # Converti in lon, lat per Polygon
                geojson_coords = [[coord[1], coord[0]] for coord in hull_points]
                poly = Polygon(geojson_coords)
                
                if poly.is_valid:
                    gdf = gpd.GeoDataFrame({'geometry': [poly]}, crs='EPSG:4326')
                    gdf_projected = gdf.to_crs('EPSG:32632')
                    area_m2 = gdf_projected.geometry.area.iloc[0]
                    superficie_km2 = area_m2 / 1_000_000
                    densita = num_immobili / superficie_km2 if superficie_km2 > 0 else 0
            except Exception:
                pass
        
        metadata_list.append({
            'cluster_id': i,
            'lat': clusters_df.loc[i, 'lat'],
            'lon': clusters_df.loc[i, 'lon'],
            'fascia_omi': majority_fascia,
            'num_immobili': num_immobili,
            'superficie_km2': round(superficie_km2, 4),
            'densita_immobili_per_km2': round(densita, 2)
        })
    
    return pd.DataFrame(metadata_list)

In [ ]:
def generate_cluster_map(coords: np.ndarray, labels: np.ndarray, centroids: np.ndarray, 
                        n_clusters: int, output_path: str) -> None:
    """
    Genera e salva la mappa dei cluster con colorazione basata sulla densità.
    """
    m = folium.Map(location=[settings.TORINO_LAT, settings.TORINO_LON], zoom_start=12)
    cluster_sizes = np.bincount(labels)
    
    # Calcola cluster validi con densità
    valid_clusters = []
    for i in range(n_clusters):
        cluster_points = coords[labels == i]
        if len(cluster_points) < 3:
            continue
        
        try:
            hull = ConvexHull(cluster_points)
            hull_points = cluster_points[hull.vertices]
            geojson_coords = [[coord[1], coord[0]] for coord in hull_points]
            poly = Polygon(geojson_coords)
            
            if poly.is_valid:
                gdf = gpd.GeoDataFrame({'geometry': [poly]}, crs='EPSG:4326')
                gdf_projected = gdf.to_crs('EPSG:32632')
                area_m2 = gdf_projected.geometry.area.iloc[0]
                area_km2 = area_m2 / 1_000_000
                density = cluster_sizes[i] / area_km2 if area_km2 > 0 else 0
                valid_clusters.append((i, poly, area_km2, density, cluster_sizes[i]))
        except Exception:
            continue
    
    # Normalizza densità e aggiungi cluster alla mappa
    densities = [d for _, _, _, d, _ in valid_clusters]
    if densities:
        norm = plt.Normalize(vmin=min(densities), vmax=max(densities))
    else:
        norm = plt.Normalize(vmin=0, vmax=1)
    
    cmap = plt.cm.viridis
    
    for i, poly, area_km2, density, num_immobili in valid_clusters:
        color = matplotlib.colors.to_hex(cmap(norm(density)))
        folium.GeoJson(
            poly.__geo_interface__,
            style_function=lambda x, c=color: {
                'fillColor': c, 
                'color': 'black', 
                'weight': 1, 
                'fillOpacity': 0.6
            },
            tooltip=f"Cluster {i}: {num_immobili} immobili, Area: {area_km2:.2f} km², Densità: {density:.2f} immobili/km²"
        ).add_to(m)
    
    # Aggiungi legenda
    if densities:
        legend_html = f'''
        <div style="position: fixed; 
                    bottom: 50px; right: 50px; width: 220px; height: 180px; 
                    background-color: white; border:2px solid grey; z-index:9999; 
                    font-size:14px; padding: 10px">
        <p style="margin: 0; font-weight: bold;">Densità Cluster</p>
        <p style="margin: 5px 0;">Min: {min(densities):.2f} immobili/km²</p>
        <p style="margin: 5px 0;">Max: {max(densities):.2f} immobili/km²</p>
        <div style="background: linear-gradient(to right, 
                    {matplotlib.colors.to_hex(cmap(0))}, 
                    {matplotlib.colors.to_hex(cmap(0.9999))}); 
                    height: 20px; margin: 10px 0;"></div>
        <div style="display: flex; justify-content: space-between;">
            <span>Bassa</span>
            <span>Alta</span>
        </div>
        <p style="margin-top: 10px; font-size: 12px;">Numero cluster: {n_clusters}</p>
        </div>
        '''
        m.get_root().html.add_child(folium.Element(legend_html))
    
    m.save(output_path)

In [ ]:
def create_quotazioni_dataset(immobili_with_omi: gpd.GeoDataFrame, labels: np.ndarray) -> pd.DataFrame:
    """
    Crea il dataset delle quotazioni medie per gruppo catastale per ogni cluster.
    
    Returns:
        DataFrame con colonne: cluster_id, gruppo_catastale, quotazione_media_al_mq
    """
    # Aggiungi cluster_id agli immobili
    immobili_with_cluster = immobili_with_omi.copy()
    immobili_with_cluster['cluster_id'] = labels
    
    # Filtra immobili con quotazione valida
    immobili_valid = immobili_with_cluster[
        immobili_with_cluster['quotazione_immobiliare_mq'].notna() & 
        immobili_with_cluster['gruppo_catastale'].notna()
    ].copy()
    
    # Converti quotazione a numerico
    immobili_valid['quotazione_immobiliare_mq'] = pd.to_numeric(
        immobili_valid['quotazione_immobiliare_mq'], 
        errors='coerce'
    )
    
    # Rimuovi valori non validi
    immobili_valid = immobili_valid[immobili_valid['quotazione_immobiliare_mq'] > 0]
    
    # Aggrega per cluster_id e gruppo_catastale
    quotazioni_agg = immobili_valid.groupby(['cluster_id', 'gruppo_catastale']).agg({
        'quotazione_immobiliare_mq': 'mean'
    }).reset_index()
    
    quotazioni_agg.columns = ['cluster_id', 'gruppo_catastale', 'quotazione_media_al_mq']
    quotazioni_agg['quotazione_media_al_mq'] = quotazioni_agg['quotazione_media_al_mq'].round(2)
    
    return quotazioni_agg

In [ ]:
# Carica immobili e prepara coordinate per il clustering
coords, immobili_with_omi, quotazioni_df = load_immobili_and_prepare_coords()
if coords is None:
    raise ValueError("Errore nel caricamento dei dati degli immobili")

print(f"Dataset caricato con successo: {len(immobili_with_omi)} immobili")
print(f"Coordinate pronte per clustering: {coords.shape}")

In [ ]:
# Carica dati POI per il conteggio delle amenity
tree, all_pois, pois_by_category = load_pois_data()
print(f"POI caricati: {len(all_pois)} punti di interesse")
print(f"Categorie POI: {list(pois_by_category.keys())}")

In [ ]:
# OPZIONALE: Calcola BSS per diversi valori di k (può richiedere tempo)
# Decommentare se si vuole esplorare diversi valori di k

# n_clusters_range = range(50, 201, 10)
# bss_values = []
# n_clusters_values = []

# print("Calcolando BSS per diversi valori di k...")
# for k in n_clusters_range:
#     kmeans_temp = KMeans(n_clusters=k, random_state=seed, n_init=10)
#     kmeans_temp.fit(coords)
    
#     # Calcola Between-Cluster Sum of Squares
#     # BSS = Total SS - Within SS
#     total_ss = np.sum((coords - coords.mean(axis=0))**2)
#     within_ss = kmeans_temp.inertia_
#     bss = total_ss - within_ss
    
#     bss_values.append(bss)
#     n_clusters_values.append(k)
#     print(f"k={k}: BSS={bss:.2f}")

# # Plot della BSS
# plt.figure(figsize=(10, 6))
# plt.plot(n_clusters_values, bss_values, marker='s', color='green', label='BSS (Between-Cluster Sum of Squares)')
# plt.title('BSS per K-Means Clustering')
# plt.xlabel('Numero di Cluster (k)')
# plt.ylabel('Between-Cluster Sum of Squares')
# plt.legend()
# plt.grid(True)
# plt.show()

print("Cella BSS plot disabilitata. K ottimale già selezionato: 150")

In [ ]:
# k_opt impostato a 150
k_opt = 150
print(f"k_opt selezionato: {k_opt}")

In [ ]:
# Esegui clustering per il k ottimale scelto
n_clusters = k_opt

print(f"\n{'='*60}")
print(f"Elaborando n_clusters = {n_clusters}")
print(f"{'='*60}")

# Crea cartelle
os.makedirs('.', exist_ok=True)

# Effettua KMeans clustering
print(f"Eseguendo KMeans clustering...")
kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
labels = kmeans.fit_predict(coords)
centroids = kmeans.cluster_centers_

# Crea dataframe dei cluster
clusters_df = pd.DataFrame({
    'cluster_id': range(n_clusters),
    'lat': centroids[:, 0],
    'lon': centroids[:, 1]
})

# ===== DATASET 1: Metadata cluster =====
print(f"Creando dataset metadata cluster...")
metadata_df = create_cluster_metadata(clusters_df, immobili_with_omi, labels, coords, n_clusters)
metadata_path = f'cluster_metadata_n_{n_clusters}.csv'
metadata_df.to_csv(metadata_path, index=False)
print(f"✓ Salvato {metadata_path}")

# ===== DATASET 2: Quotazioni per gruppo catastale =====
print(f"Creando dataset quotazioni...")
quotazioni_cluster_df = create_quotazioni_dataset(immobili_with_omi, labels)
quotazioni_path = f'cluster_quotazioni_n_{n_clusters}.csv'
quotazioni_cluster_df.to_csv(quotazioni_path, index=False)
print(f"✓ Salvato {quotazioni_path} ({len(quotazioni_cluster_df)} righe)")

# ===== DATASET 3: Conteggi amenity a diverse distanze =====
print(f"Creando dataset amenity counts...")
amenity_counts_df = count_amenities_at_distances(centroids, pois_by_category)
amenity_path = f'cluster_amenity_counts_n_{n_clusters}.csv'
amenity_counts_df.to_csv(amenity_path, index=False)
print(f"✓ Salvato {amenity_path} ({len(amenity_counts_df)} righe)")

# ===== DATASET 4: Immobili per cluster =====
print(f"Creando dataset immobili per cluster...")
immobili_cluster_df = pd.DataFrame({
    'immobile_id': immobili_with_omi['id'],
    'cluster_id': labels
})
immobili_cluster_path = f'cluster_immobili_n_{n_clusters}.csv'
immobili_cluster_df.to_csv(immobili_cluster_path, index=False)
print(f"✓ Salvato {immobili_cluster_path} ({len(immobili_cluster_df)} righe)")

# ===== Genera mappa dei cluster =====
print(f"Generando mappa cluster...")
generate_cluster_map(coords, labels, centroids, n_clusters, f'cluster_map_n_{n_clusters}.html')
print(f"✓ Salvata mappa")

print(f"\n✓ Elaborazione completata per n_clusters = {n_clusters}")
print(f"  - Metadata: {len(metadata_df)} cluster")
print(f"  - Quotazioni: {len(quotazioni_cluster_df)} righe")
print(f"  - Amenity: {len(amenity_counts_df)} righe")
print(f"  - Immobili: {len(immobili_cluster_df)} righe")

In [ ]:
# ===== Calcolo amenity_counts_n_150.csv e cluster_amenity_percentages_n_150.csv =====
import pandas as pd

# Definisci variabili necessarie
n_clusters = 150

# Carica cluster_amenity_counts_n_150.csv
cluster_amenity_df = pd.read_csv(f'cluster_amenity_counts_n_{n_clusters}.csv')

# Calcola amenity_counts_n_150.csv: totali per amenity aggregando sui cluster
amenity_totals_df = cluster_amenity_df.groupby(['category', 'amenity']).agg({
    'count_250': 'sum',
    'count_500': 'sum',
    'count_1000': 'sum',
    'count_2000': 'sum'
}).reset_index()

amenity_totals_df.columns = ['category', 'amenity', 'count_250', 'count_500', 'count_1000', 'count_2000']

# Salva amenity_counts_n_150.csv
amenity_totals_path = f'amenity_counts_n_{n_clusters}.csv'
amenity_totals_df.to_csv(amenity_totals_path, index=False)
print(f"✓ Salvato {amenity_totals_path} ({len(amenity_totals_df)} righe)")

# Calcola le percentuali: count per cluster/amenity/distance / totale per amenity/distance
# Unisci i totali al dataframe originale
cluster_amenity_with_totals = cluster_amenity_df.merge(amenity_totals_df, on=['category', 'amenity'], how='left')

# Calcola percentuali, gestendo divisione per zero
cluster_amenity_with_totals['perc_250'] = (cluster_amenity_with_totals['count_250_x'] / cluster_amenity_with_totals['count_250_y'].replace(0, 1))
cluster_amenity_with_totals['perc_500'] = (cluster_amenity_with_totals['count_500_x'] / cluster_amenity_with_totals['count_500_y'].replace(0, 1))
cluster_amenity_with_totals['perc_1000'] = (cluster_amenity_with_totals['count_1000_x'] / cluster_amenity_with_totals['count_1000_y'].replace(0, 1))
cluster_amenity_with_totals['perc_2000'] = (cluster_amenity_with_totals['count_2000_x'] / cluster_amenity_with_totals['count_2000_y'].replace(0, 1))

# Per casi dove totale è 0, imposta perc a 0
cluster_amenity_with_totals.loc[cluster_amenity_with_totals['count_250_y'] == 0, 'perc_250'] = 0
cluster_amenity_with_totals.loc[cluster_amenity_with_totals['count_500_y'] == 0, 'perc_500'] = 0
cluster_amenity_with_totals.loc[cluster_amenity_with_totals['count_1000_y'] == 0, 'perc_1000'] = 0
cluster_amenity_with_totals.loc[cluster_amenity_with_totals['count_2000_y'] == 0, 'perc_2000'] = 0

# Seleziona colonne per il file delle percentuali
percentages_df = cluster_amenity_with_totals[['cluster_id', 'category', 'amenity', 'perc_250', 'perc_500', 'perc_1000', 'perc_2000']]

# Salva cluster_amenity_percentages_n_150.csv
percentages_path = f'cluster_amenity_percentages_n_{n_clusters}.csv'
percentages_df.to_csv(percentages_path, index=False)
print(f"✓ Salvato {percentages_path} ({len(percentages_df)} righe)")

print(f"\n✓ Calcoli aggiuntivi completati per n_clusters = {n_clusters}")

In [ ]:
# ===== DATASET 4: Distanze POI per cluster =====
print(f"Creando dataset distanze POI per cluster...")
poi_distances = save_poi_distances_per_cluster(centroids, pois_by_category, max_distance_m=2000)

# Salva come JSON
poi_distances_path = f'cluster_poi_distances_n_{n_clusters}.json'
with open(poi_distances_path, 'w', encoding='utf-8') as f:
    json.dump(poi_distances, f, indent=2, ensure_ascii=False)
print(f"✓ Salvato {poi_distances_path}")

print(f"\n✓ Tutti i dataset creati per n_clusters = {n_clusters}")